# Synthetic data

> Sample inputs across the dimensions of a task

In [ ]:
#| default_exp synth

A task's synthetic inputs should cover every kind of case the model must handle, not drift towards the common ones. Following [synthetic questions for MARIS](https://fr.anckalbi.net/posts/synthetic-questions-maris/), we list the *dimensions* along which cases vary, give each a few *levels*, and sample *tuples*: combinations of one level from each dimension. Each tuple stands for one distinct kind of case. This module holds the part every task shares; each task supplies its own dimensions and a generator that turns a tuple into one input.

In [ ]:
#| export
import random
from itertools import product

In [ ]:
#| hide
from fastcore.test import *
from collections import Counter

## Tuples

Dimensions are a dict from dimension name to its levels. The MARIS dimensions make a small example:

In [ ]:
maris = dict(purpose=['scientific', 'regulatory', 'educational', 'metadata'],
             audience=['domain expert', 'professional', 'general public'],
             complexity=['direct factual', 'analytical', 'conceptual'])

In [ ]:
#| export
def tuples(dims:dict # Dimension name -> its levels
          )->list[dict]:
    "All combinations of levels, one from each dimension"
    return [dict(zip(dims, c)) for c in product(*dims.values())]

There are 4 × 3 × 3 = 36 tuples:

In [ ]:
ts = tuples(maris)
test_eq(len(ts), 36)
ts[:2]

[{'purpose': 'scientific',
  'audience': 'domain expert',
  'complexity': 'direct factual'},
 {'purpose': 'scientific',
  'audience': 'domain expert',
  'complexity': 'analytical'}]

## Sampling

`sample` calls the task's generator `gen(tuple, rng)` `n` times. It cycles through the tuples in a shuffled order, so each tuple gets `n // len(tuples)` inputs or one more: 270 inputs over 27 tuples gives exactly 10 each, and 50 inputs gives 1 or 2 each. All randomness comes from one `random.Random(seed)` passed to the generator, so a seed reproduces the whole set. Different seeds give the independent draws needed for separate training and test sets.

In [ ]:
#| export
def sample(dims:dict, # Dimension name -> its levels
           gen, # `gen(tuple, rng)` returns one input for `tuple`
           n:int, # Number of inputs
           seed:int=0 # Seed of the random generator passed to `gen`
          )->list[dict]: # Dicts with the `tuple` and the generated `input`
    "`n` inputs from `gen`, spread evenly over the tuples of `dims`"
    rng = random.Random(seed)
    ts = tuples(dims)
    rng.shuffle(ts)
    return [dict(tuple=t, input=gen(t, rng)) for t in (ts[i % len(ts)] for i in range(n))]

A generator for MARIS would ask a model for a question; this one only builds a placeholder string:

In [ ]:
def placeholder(t, rng): return f"{t['complexity']} {t['purpose']} question for the {t['audience']} (#{rng.randint(1, 99)})"

recs = sample(maris, placeholder, 72, seed=1)
recs[0]

{'tuple': {'purpose': 'metadata',
  'audience': 'general public',
  'complexity': 'analytical'},
 'input': 'analytical metadata question for the general public (#68)'}

Every tuple appears twice in 72 inputs, and the same seed gives the same inputs:

In [ ]:
test_eq(set(Counter(tuple(r['tuple'].values()) for r in recs).values()), {2})
test_eq(sample(maris, placeholder, 72, seed=1), recs)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()